In [0]:
# Null Check Script - Checks for nulls in every column of every table in a schema

# Configuration - Update these values
catalog_name = "car_workshop"
schema_name = "dim"

# Get all tables in the schema
tables = spark.sql(f"""
    SHOW TABLES IN {catalog_name}.{schema_name}
""").collect()

print(f"Found {len(tables)} tables in {catalog_name}.{schema_name}\n")

results = []

for table_row in tables:
    table_name = table_row.tableName
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    
    print(f"Checking table: {full_table_name}")
    
    # Get all columns for this table
    columns_df = spark.sql(f"DESCRIBE TABLE {full_table_name}")
    columns = [row.col_name for row in columns_df.collect() 
               if row.col_name and not row.col_name.startswith('#')]
    
    # Build a query to count nulls for each column
    null_checks = [f"SUM(CASE WHEN `{col}` IS NULL THEN 1 ELSE 0 END) as `{col}_nulls`" 
                   for col in columns]
    
    null_checks_sql = ',\n            '.join(null_checks)
    null_count_query = f"""
        SELECT 
            COUNT(*) as total_rows,
            {null_checks_sql}
        FROM {full_table_name}
    """
    
    # Execute the query
    null_counts = spark.sql(null_count_query).collect()[0]
    total_rows = null_counts['total_rows']
    
    # Collect results for columns with nulls
    for col in columns:
        null_count = null_counts[f"{col}_nulls"]
        null_percentage = (null_count / total_rows * 100) if total_rows > 0 else 0
        
        results.append({
            'table': full_table_name,
            'column': col,
            'total_rows': total_rows,
            'null_count': null_count,
            'null_percentage': round(null_percentage, 2)
        })
    
    print(f"  ✓ Checked {len(columns)} columns\n")

# Convert results to DataFrame for better visualization
from pyspark.sql import Row
results_df = spark.createDataFrame([Row(**r) for r in results])

# Show summary
print("="*80)
print("NULL CHECK SUMMARY")
print("="*80)

# Show only columns with nulls
columns_with_nulls = results_df.filter("null_count > 0").orderBy("table", "null_percentage", ascending=[True, False])

if columns_with_nulls.count() > 0:
    print(f"\nColumns with NULL values:")
    display(columns_with_nulls)
else:
    print("\n✓ No NULL values found in any column!")

# Show full results
print(f"\nComplete results (all columns):")
display(results_df.orderBy("table", "column"))